# Notebook 4: Sistema RAG Clínico Multifuente (Jufe, Stahl y CANMAT) con Embeddings y Ollama

Este notebook implementa el sistema **RAG (Retrieval-Augmented Generation)** integrando el conocimiento de las tres fuentes clínicas especializadas:
1. **Dra. Gabriela Jufe** (*Psicofarmacología Práctica*): Posología, poblaciones especiales y manejo de efectos adversos.
2. **Dr. Stephen Stahl** (*Prescriber's Guide*): Perfil de receptores, interacciones enzimáticas por citocromos (CYP450) y efectos adversos potencialmente letales.
3. **Guías CANMAT / ISBD**: Algoritmos de 1ª/2ª línea, jerarquías de evidencia y prevención de cascadas/virajes.

Además, incorpora el catálogo de **Marcas Comerciales, Laboratorios y Códigos ATC de la OMS** para responder a consultas del mundo real.

---

### Arquitectura del RAG:
```
┌─────────────────────────────────────────────────────────────┐
│ 1. Consolidación de Datasets Limpios (Jufe + Stahl + CANMAT)│
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 2. Construcción de Fichas Clínicas Estructuradas (Chunks)   │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 3. Indexación Vectorial (Embeddings con Ollama / Scikit)    │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 4. Retrieval Semántico (Top-K por Similitud de Coseno)      │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 5. Augmented Generation con LLM en Ollama (Llama 3/Mistral) │
└─────────────────────────────────────────────────────────────┘
```


In [ ]:
import os
import json
import re
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Detección dinámica y robusta de rutas
current_dir = Path.cwd()
project_root = current_dir if (current_dir / "data").exists() else current_dir.parent

DATA_PROCESSED = project_root / "data" / "processed"
print(f"Entorno configurado. Directorio de datos: {DATA_PROCESSED}")


## 1. Carga de Datasets Procesados de las Tres Fuentes

Cargamos los datasets limpios que generamos en los notebooks `01`, `02` y `03`:


In [ ]:
# 1. Cargar Jufe
df_jufe = pd.read_csv(DATA_PROCESSED / "jufe_farmacos_clean.csv")
df_jufe_efectos = pd.read_csv(DATA_PROCESSED / "jufe_efectos_adversos_clean.csv")

# 2. Cargar Stahl
df_stahl = pd.read_csv(DATA_PROCESSED / "stahl_farmacos_clean.csv")
df_stahl_eventos = pd.read_csv(DATA_PROCESSED / "stahl_eventos_adversos_clean.csv")

# 3. Cargar CANMAT
df_canmat = pd.read_csv(DATA_PROCESSED / "canmat_recomendaciones_clean.csv")
df_canmat_alertas = pd.read_csv(DATA_PROCESSED / "canmat_alertas_clinicas_clean.csv")

print(f"Jufe: {len(df_jufe)} fármacos | Stahl: {len(df_stahl)} fármacos | CANMAT: {len(df_canmat)} directrices.")


## 2. Enriquecimiento con Marcas Comerciales y Laboratorios

Para cumplir el requisito de consultar por **marca comercial** y **laboratorio fabricante**, creamos una base de homologación clínica de las marcas más prescritas en psiquiatría:


In [ ]:
CATALOGO_MARCAS = pd.DataFrame([
    {"farmaco": "Sertralina", "marca": "Zoloft", "laboratorio": "Pfizer", "pais": "Internacional", "presentacion": "Comprimidos 50mg / 100mg"},
    {"farmaco": "Sertralina", "marca": "Serlift", "laboratorio": "Roemmers", "pais": "Argentina/LatAm", "presentacion": "Comprimidos 50mg / 100mg"},
    {"farmaco": "Sertralina", "marca": "Aremis", "laboratorio": "Almirall", "pais": "España", "presentacion": "Comprimidos 50mg / 100mg"},
    {"farmaco": "Fluoxetina", "marca": "Prozac", "laboratorio": "Eli Lilly", "pais": "Internacional", "presentacion": "Cápsulas 20mg"},
    {"farmaco": "Fluoxetina", "marca": "Foxetin", "laboratorio": "Gador", "pais": "Argentina/LatAm", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Paroxetina", "marca": "Paxil / Seroxat", "laboratorio": "GlaxoSmithKline (GSK)", "pais": "Internacional", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Paroxetina", "marca": "Aropax", "laboratorio": "GSK", "pais": "LatAm", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Paroxetina", "marca": "Psicoasten", "laboratorio": "Baliarda", "pais": "Argentina", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Clozapina", "marca": "Clozaril", "laboratorio": "Novartis", "pais": "Internacional", "presentacion": "Comprimidos 25mg / 100mg"},
    {"farmaco": "Clozapina", "marca": "Lapenax", "laboratorio": "Novartis", "pais": "LatAm", "presentacion": "Comprimidos 25mg / 100mg"},
    {"farmaco": "Risperidona", "marca": "Risperdal", "laboratorio": "Janssen-Cilag", "pais": "Internacional", "presentacion": "Comprimidos 1mg / 2mg / 3mg"},
    {"farmaco": "Risperidona", "marca": "Sequinan", "laboratorio": "Roemmers", "pais": "Argentina", "presentacion": "Comprimidos 1mg / 2mg / 3mg"},
    {"farmaco": "Haloperidol", "marca": "Haldol", "laboratorio": "Janssen-Cilag", "pais": "Internacional", "presentacion": "Gotas / Comprimidos 5mg / Ampollas"},
    {"farmaco": "Litio (Carbonato de Litio)", "marca": "Plenur", "laboratorio": "Faes Farma", "pais": "España", "presentacion": "Comprimidos 400mg"},
    {"farmaco": "Litio (Carbonato de Litio)", "marca": "Ceglution", "laboratorio": "Baliarda", "pais": "Argentina", "presentacion": "Comprimidos 300mg"},
    {"farmaco": "Bupropión", "marca": "Wellbutrin / Elontril", "laboratorio": "GSK", "pais": "Internacional", "presentacion": "Comprimidos liberación prolongada 150mg / 300mg"},
    {"farmaco": "Lamotrigina", "marca": "Lamictal", "laboratorio": "GSK", "pais": "Internacional", "presentacion": "Comprimidos dispersables 25mg / 50mg / 100mg"},
    {"farmaco": "Lamotrigina", "marca": "Lamotrax", "laboratorio": "Montpellier", "pais": "Argentina", "presentacion": "Comprimidos 25mg / 50mg / 100mg"}
])

print(f"Catálogo de marcas registrado: {len(CATALOGO_MARCAS)} marcas comerciales con laboratorios.")
CATALOGO_MARCAS.head(5)


## 3. Construcción de Fichas Clínicas (Document Chunks)

En RAG, un documento de texto rico y contextualizado produce embeddings de mucha mayor calidad que fragmentos aislados. 

Construimos dos tipos de documentos:
1. **Fichas Monográficas de Fármacos:** Integran Jufe + Stahl + Marcas Comerciales + ATC.
2. **Fichas de Algoritmos y Guías Clínicas:** Integran CANMAT (líneas de recomendación, fases y contraindicaciones de viraje).


In [ ]:
documentos_rag = []

# 1. Fichas integradas de fármacos (Jufe + Stahl + Marcas)
todos_los_farmacos = sorted(list(set(df_jufe["farmaco"].tolist() + df_stahl["farmaco"].tolist())))

for f in todos_los_farmacos:
    # Obtener datos de Jufe si existen
    info_jufe = df_jufe[df_jufe["farmaco"] == f]
    # Obtener datos de Stahl si existen
    info_stahl = df_stahl[df_stahl["farmaco"] == f]
    # Obtener marcas
    marcas_f = CATALOGO_MARCAS[CATALOGO_MARCAS["farmaco"] == f]
    
    marcas_str = ", ".join([f"{r.marca} ({r.laboratorio})" for _, r in marcas_f.iterrows()]) if len(marcas_f) > 0 else "Sin marcas registradas"
    
    atc = info_jufe["atc_codigo"].values[0] if len(info_jufe) > 0 else (info_stahl["atc_codigo"].values[0] if len(info_stahl) > 0 else "N/D")
    
    contenido = f"""# FICHA FARMACOLÓGICA: {f.upper()}
- Código ATC (OMS): {atc}
- Marcas Comerciales y Laboratorios: {marcas_str}
"""
    
    if len(info_jufe) > 0:
        row_j = info_jufe.iloc[0]
        contenido += f"""
## Información Clínica (Dra. Gabriela Jufe - Psicofarmacología Práctica):
- Clase: {row_j['clase']}
- Mecanismo de Acción: {row_j['mecanismo']}
- Posología Usual: {row_j['posologia']}
- Efectos Adversos Frecuentes: {row_j['efectos_adversos']}
- Interacciones Farmacológicas: {row_j['interacciones']}
- Poblaciones Especiales: {row_j['poblaciones_especiales']}
"""

    if len(info_stahl) > 0:
        row_s = info_stahl.iloc[0]
        contenido += f"""
## Perfil Especializado y Receptores (Dr. Stephen Stahl - Guía del Prescriptor):
- Clase Stahl: {row_s['clase']}
- Perfil de Receptores: {row_s['perfil_receptores']}
- Metabolismo Citocromo P450: {row_s['metabolismo_cyp']}
- Efectos Adversos Graves / Riesgo Vital: {row_s['efectos_graves']}
- Perlas Clínicas de Stahl: {row_s['perlas_clinicas']}
"""

    documentos_rag.append({
        "id": f"farmaco_{f.lower()}",
        "tipo": "farmaco_monografia",
        "titulo": f,
        "atc": atc,
        "contenido": contenido.strip()
    })

# 2. Fichas de Guías Clínicas CANMAT
for _, row_c in df_canmat.iterrows():
    contenido_guia = f"""# GUÍA CLÍNICA CANMAT: {row_c['patologia'].upper()} ({row_c['fase'].upper()})
- Recomendación: {row_c['linea']}
- Estrategia: {row_c['estrategia']}
- Fármaco / Esquema: {row_c['farmaco']} (ATC: {row_c['atc']})
- Nivel de Evidencia Científica: {row_c['nivel_evidencia']}
- Directriz Clínica: {row_c['comentarios']}
- Fuente: Canadian Network for Mood and Anxiety Treatments (CANMAT)
"""
    documentos_rag.append({
        "id": f"canmat_{row_c['patologia'][:3].lower()}_{row_c['farmaco'].split()[0].lower()}",
        "tipo": "guia_canmat",
        "titulo": f"CANMAT: {row_c['patologia']} - {row_c['farmaco']}",
        "atc": row_c['atc'],
        "contenido": contenido_guia.strip()
    })

print(f"Total de documentos indexables en el RAG: {len(documentos_rag)}")


## 4. Motor de Embeddings y Almacén Vectorial

Diseñamos una clase `VectorStore` híbrida:
* **Conexión nativa a Ollama:** Intenta generar embeddings mediante el endpoint local `http://localhost:11434/api/embeddings` (modelo `nomic-embed-text` o `bge-m3`).
* **Fallback determinista con Scikit-Learn (TF-IDF Semántico):** Si Ollama no está iniciado en segundo plano, utiliza un vectorizador denso local para que el notebook funcione de inmediato.


In [ ]:
class ClinicalVectorStore:
    def __init__(self, ollama_host="http://localhost:11434", embedding_model="embeddinggemma:latest"):
        self.ollama_host = ollama_host
        self.embedding_model = embedding_model
        self.documentos = []
        self.embeddings = None
        self.use_ollama = False
        self._verificar_ollama()
        
    def _verificar_ollama(self):
        try:
            r = requests.get(f"{self.ollama_host}/api/tags", timeout=1.5)
            if r.status_code == 200:
                self.use_ollama = True
                print(f"✓ Conectado a Ollama local ({self.ollama_host}).")
            else:
                self.use_ollama = False
        except Exception:
            self.use_ollama = False
            print("ℹ Ollama no está activo en localhost:11434. Se usará el vectorizador semántico local de Scikit-Learn.")

    def indexar_documentos(self, docs):
        self.documentos = docs
        textos = [d["contenido"] for d in docs]
        
        if self.use_ollama:
            try:
                emb_list = []
                for t in textos:
                    res = requests.post(
                        f"{self.ollama_host}/api/embeddings",
                        json={"model": self.embedding_model, "prompt": t},
                        timeout=10
                    )
                    emb_list.append(res.json()["embedding"])
                self.embeddings = np.array(emb_list)
                print(f"Embeddings calculados con Ollama ({self.embedding_model}).")
                return
            except Exception as e:
                print(f"Error generando embeddings en Ollama: {e}. Usando fallback Scikit-Learn.")
                
        # Fallback local con vectorizador TF-IDF con n-gramas
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
        self.embeddings = self.vectorizer.fit_transform(textos).toarray()
        print(f"Indexados {len(self.documentos)} documentos con vectorizador local.")

    def buscar(self, query, top_k=3):
        """Recupera los top_k documentos más relevantes para una consulta."""
        if self.use_ollama:
            try:
                res = requests.post(
                    f"{self.ollama_host}/api/embeddings",
                    json={"model": self.embedding_model, "prompt": query},
                    timeout=5
                )
                query_vec = np.array(res.json()["embedding"]).reshape(1, -1)
                sims = cosine_similarity(query_vec, self.embeddings)[0]
            except Exception:
                query_vec = self.vectorizer.transform([query]).toarray()
                sims = cosine_similarity(query_vec, self.embeddings)[0]
        else:
            query_vec = self.vectorizer.transform([query]).toarray()
            sims = cosine_similarity(query_vec, self.embeddings)[0]
            
        indices_top = np.argsort(sims)[::-1][:top_k]
        
        resultados = []
        for idx in indices_top:
            doc = self.documentos[idx].copy()
            doc["score_similitud"] = round(float(sims[idx]), 4)
            resultados.append(doc)
            
        return resultados

# Inicializar e indexar
vector_store = ClinicalVectorStore()
vector_store.indexar_documentos(documentos_rag)


## 5. Módulo de Generación Asistida (RAG Prompting & Ollama)

Construimos el generador clínico que toma los documentos recuperados por el vector store, construye un prompt con directrices de seguridad médica, y llama a Ollama para redactar la respuesta razonada.


In [ ]:
PROMPT_SISTEMA_CLINICO = """Eres un Asistente Clínico Especialista en Farmacología Psiquiátrica y Polifarmacia.
Tu tarea es responder preguntas médicas basándote EXCLUSIVAMENTE en el contexto recuperado de las tres fuentes oficiales:
1. Dra. Gabriela Jufe (Psicofarmacología Práctica)
2. Dr. Stephen Stahl (Prescriber's Guide)
3. Guías CANMAT / ISBD para trastornos del ánimo

REGLAS DE SEGURIDAD:
- Cita siempre la fuente médica que avala cada afirmación (ej: 'Según Jufe...', 'De acuerdo con Stahl...', 'La guía CANMAT indica...').
- Si la pregunta involucra marcas comerciales o laboratorios, menciona la correspondencia exacta con el principio activo.
- Si detectas una posible cascada de prescripción o interacción grave (ej. citocromo CYP2D6, síndrome serotoninérgico, síntomas extrapiramidales), resáltala como una ADVERTENCIA CLÍNICA prioritaria.
- No inventes interacciones no descritas en el contexto. Si falta información, declara la limitación.
"""

def generar_respuesta_rag(pregunta, vector_store, modelo_ollama="qwen3.5:2b-mlx", top_k=2):
    # 1. Recuperar contexto relevante
    docs_recuperados = vector_store.buscar(pregunta, top_k=top_k)
    
    contexto_str = "\n\n---\n\n".join([d["contenido"] for d in docs_recuperados])
    
    prompt_completo = f"""{PROMPT_SISTEMA_CLINICO}

CONTEXTO CLÍNICO RECUPERADO:
{contexto_str}

PREGUNTA DEL MÉDICO / USUARIO:
{pregunta}

RESPUESTA CLÍNICA ESTRUCTURADA:"""

    # 2. Consultar a Ollama
    if vector_store.use_ollama:
        try:
            res = requests.post(
                f"{vector_store.ollama_host}/api/generate",
                json={
                    "model": modelo_ollama,
                    "prompt": prompt_completo,
                    "stream": False
                },
                timeout=45
            )
            return res.json()["response"], docs_recuperados
        except Exception as e:
            print(f"Aviso: No se pudo contactar con el modelo en Ollama ({e}). Mostrando síntesis clínica local.")

    # Fallback: Síntesis determinista basada en el contexto recuperado
    sintesis = f"""[SÍNTESIS CLÍNICA BASADA EN DOCUMENTOS RECUPERADOS]
Para la consulta: '{pregunta}'

Documentos relevantes encontrados en la base de conocimiento:
"""
    for d in docs_recuperados:
        sintesis += f"\n* [{d['titulo']}] (Relevancia: {d['score_similitud']})\n"
        # Extraer líneas principales
        lineas = [l for l in d['contenido'].split('\n') if ':' in l][:5]
        sintesis += '\n'.join(['  ' + l for l in lineas]) + '\n'
        
    sintesis += "\nNota: Inicia Ollama con `ollama run llama3.2` para obtener la redacción clínica fluida del LLM."
    return sintesis, docs_recuperados


## 6. Pruebas y Casos de Uso del RAG

A continuación, ejecutamos las 4 consultas clave del proyecto final para verificar la recuperación semántica y la respuesta médica:

---

### Caso 1: Principio activo, marcas comerciales y laboratorios


In [ ]:
pregunta_1 = "¿Qué laboratorios fabrican la Sertralina y cuáles son sus nombres comerciales y posología según Jufe?"
respuesta, fuentes = generar_respuesta_rag(pregunta_1, vector_store)
print(respuesta)


### 📋 Respuesta Clínica Registrada (Caso 1)
> **Documento Recuperado:** `Sertralina` (Similitud: `0.666`)  
> **Fuente Médica:** Dra. Gabriela Jufe (*Psicofarmacología Práctica*)

**Respuesta Clínica**

Basado en la ficha farmacológica y la información clínica proporcionada:

1.  **Laboratorios y Marcas Comerciales:**
    *   **Pfizer:** Fabrica el fármaco bajo la marca comercial **Zoloft**.
    *   **Roemmers:** Fabrica el fármaco bajo la marca comercial **Serlift**.
    *   **Almirall:** Fabrica el fármaco bajo la marca comercial **Aremis**.

2.  **Posología (Dosis de Administración):**
    *   **Dosis Inicial (Inicio):** 25-50 mg/día.
    *   **Rango Terapéutico:** 50-200 mg/día.

**Fuente Médica:** Dra. Gabriela Jufe - Psicofarmacología Práctica


### Caso 2: Interacciones enzimáticas de citocromos (CYP450) según Stahl


In [ ]:
pregunta_2 = "¿Qué interacciones y precauciones enzimáticas con CYP2D6 describe Stahl para Paroxetina y Bupropión?"
respuesta, fuentes = generar_respuesta_rag(pregunta_2, vector_store)
print(respuesta)


### 📋 Respuesta Clínica Registrada (Caso 2)
> **Documento Recuperado:** `Bupropión` (Similitud: `0.558`)  
> **Fuente Médica:** Dr. Stephen Stahl (*Prescriber's Guide*)

Basado en la información clínica provista y el perfil de Stahl (Clase IRND) para Bupropión, junto con las indicaciones estándar de Farmacología Clínica derivadas del Jefe y Canmat (CANMAT), aquí se presenta el análisis:

**RESPUESTA CLÍNICA:**

Sí existe una interacción relevante entre Paroxetina (Paroxetina) y Bupropión debido a la acción enzimática compartida sobre las enzimas citocromo CYP. Aunque Stahl no detalla específicamente la co-efecto "Paroxetin + Bupropion", sus manuales de prescripción señalan la precaución por el metabolismo enzimático de ambas fármacos.

### 1. Interacciones y Precauciones enzimáticas según Stahl:
El perfil especializado del **Stahl** indica que ambos medicamentos son metabolizados por enzimas del citocromo P450 (CYP). La interacción principal se basa en el hecho de que **Bupropión es un inhibitoro potente de CYP2D6**, mientras que Paroxetina depende en gran parte de este sistema para su descomposición.

### 2. Precaución:
*   **Riesgo:** Con la co-administración, el riesgo de **acumulación** de Paroxetina aumenta debido a la inhibición de CYP2D6 por Bupropión, mientras que el Bupropión puede aumentar su toxicidad si se metaboliza por vías normales sin la descomposición.
*   **Estimación:** En casos clínicos donde ambos están prescritos, es necesario monitorear los niveles de paroxetina y asegurar una adecuada separación del tiempo (generalmente más de 1 semana) o ajustar dosis si hay historia de abuso de alcohol/medicamentos, siguiendo las advertencias de Jefe en la clínica.

### 3. Referencia:
*   **Stahl:** *"Bupropión es un potente inhibidor de CYP2D6"*.
*   **Jefe/Stahl:** La interacción debe ser evaluada con el cuidado debido a la posible alteración de la farmacocinética de Paroxetina.

---
**Nota Clínica:**
Se recomienda consultar siempre los manuales de Stahl y las recomendaciones del Jefe sobre la co-administración de antidepresivos y iróns (Bupropión) para evitar sobredosis o eventos adversos graves, especialmente en pacientes con antecedentes de epilepsia (risco convulsivo) y paroxetina.


### Caso 3: Guías CANMAT - Tratamiento y Contraindicación de Viraje


In [ ]:
pregunta_3 = "¿Qué fármacos recomienda CANMAT en primera línea para manía aguda y por qué desaconseja los antidepresivos solos en depresión bipolar?"
respuesta, fuentes = generar_respuesta_rag(pregunta_3, vector_store)
print(respuesta)


### 📋 Respuesta Clínica Registrada (Caso 3)
> **Documento Recuperado:** `CANMAT: Trastorno Bipolar - Litio + Antipsicótico Atípico` (Similitud: `0.7282`)  
> **Fuente Médica:** Guías CANMAT / ISBD (*Trastornos del Ánimo*)

**Respuesta Clínica Directa:**

Según la **Guía Clínica CANMAT**, el tratamiento de primera línea para **manía aguda** en trastorno bipolar implica la combinación de dos fármacos clave: **Litio + Antipsicótico Atípico**.

La estrategia recomendada se fundamenta en una evidencia científica con **nivel 1 (Level 1)**, que establece claramente que esta estrategia ofrece mayor tasa de respuesta en casos severos de manía comparada con la monoterapia.

### Por qué no se desaconsejan los antidepresivos solos en depresión bipolar:

El canonamiento de CANMAT en su Directriz Clínica indica explícitamente que **desaconseja los antidepresivos solos (monoterapia)** para el tratamiento de la depresión bipolar. La razón principal es la alta tasa de riesgo asociado al uso de estos fármacos sin antipsicotico o litio en pacientes con trastorno bipolar, lo que podría llevar a:
1.  **Inducción de manía:** Uso incorrecto del antidepresivo como tratamiento primario puede precipitar el inicio de manía o ciclos recurrentes más graves.
2.  **Desregulación del ciclo:** En pacientes con depresión y bipolaridad, el uso aislado de estos fármacos es generalmente ineficaz para mantener la estabilidad emocional en comparación con combinaciones complejas que incluyen litio.

*Fuente:* Canadian Network for Mood and Anxiety Treatments (CANMAT), *Guía Clínica*, Nivel 1, Estrategia Combinada (Litio + Antipsicótico).


### Caso 4: Detección de Cascada de Prescripción (Haloperidol y Biperideno)


In [ ]:
pregunta_4 = "Un paciente en tratamiento con Haloperidol comienza con rigidez y temblor y le indican medicación antiparkinsoniana. ¿Qué dicen Jufe y Stahl sobre este riesgo?"
respuesta, fuentes = generar_respuesta_rag(pregunta_4, vector_store)
print(respuesta)


### 📋 Respuesta Clínica Registrada (Caso 4)
> **Documento Recuperado:** `Haloperidol` (Similitud: `0.5781`)  
> **Fuente Médica:** Jufe & Stahl (*Riesgo de Cascada de Prescripción*)

**RESPUESTA CLÍNICA:**

Según la información proporcionada por **Dra. Gabriela Jufe** (Jufe), existe un **alto riesgo de inducir cascadas de medicación en geriatría**. Este efecto es debido a la **confusión entre el Síndrome del Parkinson Idiopático y las SEPs** (Serotonergias o interacciones complejas) con los antipsicóticos.

En el contexto general de farmacología, **Stahl (Stahl)** confirma que cuando se unen antipsicóticos típicos (como Haloperidol) con medicamentos antiparkinsonianos en ancianos, se aumenta drásticamente el riesgo de complicaciones por confusión diagnóstica.

Citas médicas:
*   **Jufe:** "Poblaciones Especiales: Alto riesgo de inducir cascadas de medicación en geriatría por confusión de SEP con Parkinson idiopático." (Dra. Gabriela Jufe - Psicofarmacología Práctica).
*   **Stahl:** Confirma que la interacción entre antipsicóticos y fármacos para Parkinson en adultos mayores genera una alta probabilidad de complicaciones farmacológicas debido a los efectos secundarios acumulados (Parkinsonismo, discinesia tardía) y riesgo de confusiones diagnósticas.


## 7. Exportación de la Base de Conocimiento a JSON / SQLite

Guardamos las fichas clínicas integradas en `data/processed/rag_knowledge_base.json` para que puedan ser cargadas directamente por tu aplicación de **Streamlit**.


In [ ]:
out_rag_json = DATA_PROCESSED / "rag_knowledge_base.json"

with open(out_rag_json, "w", encoding="utf-8") as f:
    json.dump(documentos_rag, f, indent=2, ensure_ascii=False)

print(f"Base de conocimiento RAG exportada con éxito ({len(documentos_rag)} documentos) en: {out_rag_json}")
